# ChemAI: Predict the Cure

**Команда 29 - noname**

Задача: построить ML-pipeline и предсказать три целевые переменные:
* `IC50, mM` — концентрация, при которой вещество подавляет 50% активности вируса;
* `CC50, mM` — концентрация, при которой вещество токсично для 50% клеток;
* `SI` — индекс селективности.

Метрика оценки: **RMSE**, усреднённый по трём таргетам.

> Важно: `SI` связан с `IC50` и `CC50`, однако по условию соревнования его необходимо предсказывать как **отдельную переменную**.

## Шаг 1. Подготовка окружения

Подключаем репозиторий проекта к Google Colab и переходим в рабочую папку.

Если репозиторий уже скачан - повторное клонирование пропускается.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/drhommie/ChemAI-Predict-the-Cure-29_noname.git"
REPO_NAME = "ChemAI-Predict-the-Cure-29_noname"

%cd /content

if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}
else:
    print("Репозиторий уже склонирован")

%cd /content/{REPO_NAME}
!ls

### Вывод

Окружение готово. Мы работаем в папке проекта внутри Colab.

---

### Общий вывод по Шагу 1

Репозиторий подключён. В проекте есть папки `data`, `notebooks`, `src`, `submissions`.

## Шаг 2. Загрузка данных

Данные хакатона закрытые и не хранятся в публичном репозитории.

Скачиваем три файла из Google Drive в папку `data`:
* `train.csv`;
* `test.csv`;
* `sample_submission.csv`.

**Ссылка на данные вставляется вручную перед запуском - она не хранится в коде репозитория.**

In [ ]:
from pathlib import Path

DRIVE_FOLDER_URL = "PASTE_GOOGLE_DRIVE_FOLDER_LINK_HERE"

if DRIVE_FOLDER_URL == "PASTE_GOOGLE_DRIVE_FOLDER_LINK_HERE":
    raise ValueError("Вставьте ссылку на Google Drive папку с данными в переменную DRIVE_FOLDER_URL")

Path("data").mkdir(exist_ok=True)

!pip -q install gdown
!gdown --folder "{DRIVE_FOLDER_URL}" -O data --remaining-ok

required_files = [
    "data/train.csv",
    "data/test.csv",
    "data/sample_submission.csv",
]

missing_files = []

for file_path in required_files:
    if Path(file_path).exists():
        print(f"OK: {file_path}")
    else:
        missing_files.append(file_path)

if missing_files:
    raise FileNotFoundError(f"Не найдены файлы: {missing_files}")

print("Все файлы найдены")

### Вывод

Все нужные файлы найдены в папке `data`.

---

### Общий вывод по Шагу 2

Данные скачаны и готовы к чтению.   
CSV-файлы не добавляются в репозиторий - они доступны только в рабочей среде Colab.

## Шаг 3. Чтение данных

Загружаем три таблицы через `pandas.read_csv()`.

* `train` - обучающая выборка с известными ответами;
* `test` - тестовая выборка, для которой нужно сделать предсказания;
* `sample_submission` - шаблон файла для загрузки на Kaggle.

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

### Вывод

Данные загружены:
* `train`: 751 строка, 214 колонок;
* `test`: 250 строк, 211 колонок;
* `sample_submission`: 250 строк, 4 колонки.

В `train` на 3 колонки больше - это целевые переменные.

---

### Общий вывод по Шагу 3

Данные успешно прочитаны. Размеры таблиц соответствуют описанию задачи.

## Шаг 4. Исследование данных

Смотрим на структуру данных: первые строки, типы колонок, целевые переменные.

Это нужно, чтобы понять с чем мы работаем и правильно разделить признаки и таргеты.

### Шаг 4.1. Просмотр первых строк

Смотрим первые строки каждой таблицы через `.head()`, чтобы понять структуру данных.

In [ ]:
display(train.head())
display(test.head())
display(sample_submission.head())

#### Вывод

В `train` видны колонки `index`, три целевые переменные и молекулярные признаки.  
В `test` целевых переменных нет.  
В `sample_submission` показан нужный формат ответа: `index`, `IC50`, `CC50`, `SI`.

### Шаг 4.2. Первичная проверка структуры

Проверяем:
* список целевых переменных;
* количество колонок в каждой таблице;
* совпадают ли признаки в `train` и `test`.

In [ ]:
target_cols = ["IC50, mM", "CC50, mM", "SI"]

train_features = [col for col in train.columns if col not in target_cols]
test_features = list(test.columns)

print("Целевые переменные:")
print(target_cols)

print("\nКоличество колонок:")
print("train:", train.shape[1])
print("test:", test.shape[1])
print("sample_submission:", sample_submission.shape[1])

print("\nКоличество признаков без таргетов:")
print("train:", len(train_features))
print("test:", len(test_features))

print("\nПризнаки train и test совпадают:", train_features == test_features)

print("\nКолонки sample_submission:")
print(list(sample_submission.columns))

#### Вывод

* Целевых переменных: 3 (`IC50, mM`, `CC50, mM`, `SI`);
* Колонок в `train`: 214, в `test`: 211;
* Признаки в `train` и `test` совпадают.

Формат submission: `index`, `IC50`, `CC50`, `SI`.

### Шаг 4.3. Проверка пропусков

Большинство моделей не умеют работать с пропущенными значениями - их нужно заполнить заранее.

Проверяем количество пропусков в `train` и `test`.

In [ ]:
train_missing = train.isna().sum()
test_missing = test.isna().sum()

train_missing = train_missing[train_missing > 0].sort_values(ascending=False)
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

print("Количество колонок с пропусками в train:", len(train_missing))
print("Количество колонок с пропусками в test:", len(test_missing))

print("\nПропуски в train:")
display(train_missing)

print("\nПропуски в test:")
display(test_missing)

print("\nОбщее количество пропусков в train:", train.isna().sum().sum())
print("Общее количество пропусков в test:", test.isna().sum().sum())

#### Вывод

В данных есть немного пропусков:
* `train`: 24 пропуска в 12 признаках;
* `test`: 12 пропусков в тех же признаках.

Дальше заполним их медианой, посчитанной только по обучающей выборке.

### Шаг 4.4. Проверка дубликатов

Проверяем, есть ли полностью одинаковые строки. Дубликаты могут мешать обучению.

In [ ]:
print("Дубликаты строк в train:", train.duplicated().sum())
print("Дубликаты строк в test:", test.duplicated().sum())

#### Вывод

Дубликатов нет ни в `train`, ни в `test`. Удалять строки не нужно.

---

### Общий вывод по Шагу 4

Мы изучили структуру данных:
* в `train` 3 целевые переменные и 210 признаков;
* признаки в `train` и `test` совпадают;
* в данных есть небольшое количество пропусков (24 в train, 12 в test);
* дубликатов нет.

Данные можно передавать в модель после заполнения пропусков.

## Шаг 5. Подготовка признаков и таргетов

Фиксируем список целевых переменных и признаков.

Колонку `index` не берём в обучение - это просто номер строки, а не химическое свойство молекулы.

In [ ]:
target_cols = ["IC50, mM", "CC50, mM", "SI"]
submission_target_cols = ["IC50", "CC50", "SI"]
id_col = "index"

feature_cols = [
    col for col in train.columns
    if col not in target_cols and col != id_col
]

missing_in_test = sorted(set(feature_cols) - set(test.columns))
extra_in_test = sorted(set(test.columns) - set(feature_cols) - {id_col})

print("Количество таргетов:", len(target_cols))
print("Количество признаков:", len(feature_cols))
print("Признаков из train, которых нет в test:", len(missing_in_test))
print("Лишних признаков в test:", len(extra_in_test))

### Вывод

* Целевых переменных: 3;
* Признаков: 210;
* `index` не используется как признак;
* Признаки в `train` и `test` совпадают.

---

### Общий вывод по Шагу 5

Признаки и таргеты зафиксированы. Никаких расхождений между `train` и `test` нет.

## Шаг 6. Формируем X, y и X_test

Разделяем данные на признаки и целевые переменные.

* `X` - признаки для обучения;
* `y` - целевые переменные (IC50, CC50, SI);
* `X_test` - признаки тестовой выборки.

### Шаг 6.1. Создаём X, y, X_test

In [ ]:
X = train[feature_cols]
y = train[target_cols]

X_test = test[feature_cols]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

#### Вывод

* `X`: 751 объект, 210 признаков;
* `y`: 751 объект, 3 таргета;
* `X_test`: 250 объектов, 210 признаков.

### Шаг 6.2. Проверка типов признаков

Убеждаемся, что все признаки числовые - иначе их нужно кодировать.

In [ ]:
non_numeric_cols = X.select_dtypes(exclude="number").columns.tolist()

if non_numeric_cols:
    print("Нечисловые признаки:", non_numeric_cols)
else:
    print("Все признаки числовые")

#### Вывод

Все признаки числовые. Кодирование не требуется.

---

### Общий вывод по Шагу 6

X, y и X_test сформированы. Все признаки числовые - можно передавать в модель.

## Шаг 7. Разделение на train и validation

Делим обучающую выборку на две части:
* `X_train`, `y_train` - для обучения модели;
* `X_valid`, `y_valid` - для проверки качества.

Это нужно, чтобы сравнивать модели локально до загрузки на Kaggle.

Используем пропорцию 80/20 и фиксированный `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_train shape:", y_train.shape)
print("y_valid shape:", y_valid.shape)

### Вывод

Данные разделены:
* `X_train`: 600 объектов, 210 признаков;
* `X_valid`: 151 объект, 210 признаков.

`random_state=42` зафиксирован - результат можно повторить.

---

### Общий вывод по Шагу 7

Train/validation split выполнен в пропорции 80/20.

## Шаг 8. Первая baseline-модель: LinearRegression

Сначала строим простую baseline-модель - она нужна как первая точка отсчёта.

Все следующие модели будем сравнивать с ней.

### Шаг 8.1. Заполнение пропусков медианой

Большинство моделей не принимают пропуски на вход.

Заполняем медианой, посчитанной **только по `X_train`**. Это важно: нельзя использовать статистики из validation или test — иначе будет утечка данных.

К validation и test применяем уже готовое значение медианы - только `.fillna()`.

In [ ]:
print("Пропуски до заполнения:")
print("X_train:", X_train.isna().sum().sum())
print("X_valid:", X_valid.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

train_medians = X_train.median()

X_train_filled = X_train.fillna(train_medians)
X_valid_filled = X_valid.fillna(train_medians)
X_test_filled = X_test.fillna(train_medians)

print("\nПропуски после заполнения:")
print("X_train_filled:", X_train_filled.isna().sum().sum())
print("X_valid_filled:", X_valid_filled.isna().sum().sum())
print("X_test_filled:", X_test_filled.isna().sum().sum())

#### Вывод

Пропуски заполнены медианой. До заполнения было 12 пропусков в каждой части - теперь 0.

### Шаг 8.2. Обучаем LinearRegression

Начнём с простой модели — линейной регрессии.  
Она ищет линейную зависимость между признаками и целевой переменной.  
Используем её как отправную точку для сравнения с более сложными моделями.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_filled, y_train)

valid_predictions = model.predict(X_valid_filled)

print("Предсказания на validation готовы")
print("Размер предсказаний:", valid_predictions.shape)

#### Вывод

Модель обучена на train. Мы получили предсказания для 151 объекта validation
по трём таргетам — IC50, CC50 и SI.

### Шаг 8.3. Считаем RMSE на validation

RMSE (корень из среднеквадратичной ошибки) - метрика соревнования.

Чем меньше значение, тем лучше модель.

Считаем RMSE отдельно для каждого таргета и берём среднее.

In [ ]:
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def get_scores(y_true, predictions):
    ic50 = rmse(y_true["IC50, mM"], predictions[:, 0])
    cc50 = rmse(y_true["CC50, mM"], predictions[:, 1])
    si = rmse(y_true["SI"], predictions[:, 2])
    mean_score = np.mean([ic50, cc50, si])
    return ic50, cc50, si, mean_score

def print_scores(title, ic50, cc50, si, mean_score):
    print(title)
    print(f"IC50, mM: {ic50:.5f}")
    print(f"CC50, mM: {cc50:.5f}")
    print(f"SI: {si:.5f}")
    print(f"Средняя RMSE: {mean_score:.5f}")

score_ic50_base, score_cc50_base, score_si_base, mean_rmse = get_scores(
    y_valid,
    valid_predictions
)

print_scores(
    "RMSE по таргетам:",
    score_ic50_base,
    score_cc50_base,
    score_si_base,
    mean_rmse
)

def make_result_table(rows):
    columns = ["model", "IC50_RMSE", "CC50_RMSE", "SI_RMSE", "mean_RMSE"]
    return pd.DataFrame(rows, columns=columns).sort_values("mean_RMSE")

#### Вывод

Baseline на `LinearRegression`:
* `IC50, mM`: 458.34621;
* `CC50, mM`: 851.62080;
* `SI`: 405.96383;
* **средняя RMSE: 571.97695**.

Ошибка большая. Линейная регрессия плохо работает на этих данных.

### Шаг 8.4. Обучаем baseline на всём train

Теперь обучаем модель на всех обучающих данных и делаем предсказания для `test`.

Медиану для заполнения пропусков считаем по полному `X`.

In [ ]:
full_train_medians = X.median()

X_filled = X.fillna(full_train_medians)
X_test_filled_final = X_test.fillna(full_train_medians)

final_model = LinearRegression()

final_model.fit(X_filled, y)

test_predictions = final_model.predict(X_test_filled_final)

print("Предсказания для test готовы")
print("Размер test_predictions:", test_predictions.shape)

#### Вывод

Модель обучена на 751 объекте. Размер предсказаний для test: `(250, 3)` - верно.

### Шаг 8.5. Создаём baseline submission

Собираем предсказания в файл формата `sample_submission`.

`sample_submission` используем только как шаблон - никакие данные из него не подсматриваем.

In [ ]:
submission = sample_submission.copy()

submission["IC50"] = test_predictions[:, 0]
submission["CC50"] = test_predictions[:, 1]
submission["SI"] = test_predictions[:, 2]

display(submission.head())

print("Размер submission:", submission.shape)
print("Колонки submission:", submission.columns.tolist())

assert submission.shape == sample_submission.shape
assert submission.columns.tolist() == sample_submission.columns.tolist()

print("Формат submission совпадает с sample_submission")

#### Вывод

Файл создан. Размер: `(250, 4)`, колонки: `index`, `IC50`, `CC50`, `SI`.

### Шаг 8.6. Сохраняем baseline submission

In [ ]:
from pathlib import Path

submissions_dir = Path("submissions")
submissions_dir.mkdir(exist_ok=True)

submission_path = submissions_dir / "baseline_linear_regression_submission.csv"

submission.to_csv(submission_path, index=False)

if submission_path.exists():
    print("Файл сохранён:", submission_path)
    print("Размер файла:", submission_path.stat().st_size, "байт")
else:
    print("Файл не сохранён")

#### Вывод

Файл сохранён: `submissions/baseline_linear_regression_submission.csv`

---

### Общий вывод по Шагу 8

Мы прошли первый полный цикл: загрузка данных — обработка — обучение — submission.

Результат baseline на validation:
* `IC50, mM`: 458.35, `CC50, mM`: 851.62, `SI`: 405.96;
* **средняя RMSE: 571.977**.

Качество слабое — модель даёт отрицательные предсказания, что некорректно для химических показателей.
В следующих шагах попробуем более сложную модель.

## Шаг 9. Анализ ошибок baseline

Смотрим, где baseline ошибается сильнее всего, и проверяем наличие отрицательных предсказаний.

### Шаг 9.1. Считаем ошибки по каждому объекту

In [ ]:
valid_result = y_valid.copy()

valid_result["pred_IC50"] = valid_predictions[:, 0]
valid_result["pred_CC50"] = valid_predictions[:, 1]
valid_result["pred_SI"] = valid_predictions[:, 2]

valid_result["error_IC50"] = valid_result["IC50, mM"] - valid_result["pred_IC50"]
valid_result["error_CC50"] = valid_result["CC50, mM"] - valid_result["pred_CC50"]
valid_result["error_SI"] = valid_result["SI"] - valid_result["pred_SI"]

valid_result.head()

#### Вывод

В `valid_result` добавлены реальные значения, предсказания и ошибки по трём таргетам.

### Шаг 9.2. Проверяем отрицательные предсказания

Химические показатели `IC50`, `CC50`, `SI` не должны быть отрицательными по смыслу.

In [ ]:
pred_cols = ["pred_IC50", "pred_CC50", "pred_SI"]

print("Минимальные предсказанные значения:")
print(valid_result[pred_cols].min())

print("\nКоличество отрицательных предсказаний:")
print((valid_result[pred_cols] < 0).sum())

#### Вывод

У baseline есть отрицательные предсказания:
* `IC50`: 33, `CC50`: 18, `SI`: 57.

Это проблема линейной регрессии - она не ограничивает область значений.

### Шаг 9.3. Самые большие ошибки baseline

In [ ]:
valid_result["abs_error_IC50"] = valid_result["error_IC50"].abs()
valid_result["abs_error_CC50"] = valid_result["error_CC50"].abs()
valid_result["abs_error_SI"] = valid_result["error_SI"].abs()

print("Самые большие ошибки по IC50:")
display(valid_result.sort_values("abs_error_IC50", ascending=False).head())

print("Самые большие ошибки по CC50:")
display(valid_result.sort_values("abs_error_CC50", ascending=False).head())

print("Самые большие ошибки по SI:")
display(valid_result.sort_values("abs_error_SI", ascending=False).head())

#### Вывод

На отдельных объектах ошибки очень большие:
* по `IC50` есть ошибка > 3400;
* по `CC50` есть ошибка > 7600.

Из-за таких выбросов RMSE сильно растёт.

---

### Общий вывод по Шагу 9

Baseline работает, но у него три проблемы:
1. Отрицательные предсказания (`IC50` - 33, `CC50` - 18, `SI` - 57).
2. Очень большие ошибки на отдельных объектах.
3. Слишком простая модель для данных с нелинейными зависимостями.

Дальше пробуем `RandomForestRegressor`.

## Шаг 10. RandomForestRegressor

Случайный лес строит много деревьев решений и усредняет их предсказания - это помогает находить нелинейные зависимости лучше, чем линейная регрессия.

Сначала обучаем одну общую модель на все три таргета.

### Шаг 10.1. Обучаем RandomForestRegressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_filled, y_train)

rf_valid_predictions = rf_model.predict(X_valid_filled)

score_ic50, score_cc50, score_si, rf_mean_score = get_scores(
    y_valid,
    rf_valid_predictions
)

print_scores(
    "RandomForestRegressor validation RMSE:",
    score_ic50,
    score_cc50,
    score_si,
    rf_mean_score
)

#### Вывод

RandomForest обучен. Результат на validation:
* `IC50, mM`: 383.46, `CC50, mM`: 431.36, `SI`: 293.57;
* **средняя RMSE: 369.46**.

### Шаг 10.2. Проверяем отрицательные предсказания

In [ ]:
rf_valid_result = y_valid.copy()

rf_valid_result["pred_IC50"] = rf_valid_predictions[:, 0]
rf_valid_result["pred_CC50"] = rf_valid_predictions[:, 1]
rf_valid_result["pred_SI"] = rf_valid_predictions[:, 2]

rf_pred_cols = ["pred_IC50", "pred_CC50", "pred_SI"]

print("Минимальные предсказанные значения RandomForest:")
print(rf_valid_result[rf_pred_cols].min())

print("\nКоличество отрицательных предсказаний RandomForest:")
print((rf_valid_result[rf_pred_cols] < 0).sum())

#### Вывод

Отрицательных предсказаний нет - RandomForest не выдаёт их для этих данных.

### Шаг 10.3. Сравниваем с baseline

In [ ]:
models_result = make_result_table([
    ["LinearRegression", score_ic50_base, score_cc50_base, score_si_base, mean_rmse],
    ["RandomForestRegressor", score_ic50, score_cc50, score_si, rf_mean_score],
])

models_result

#### Вывод

`RandomForestRegressor` заметно лучше baseline:
* LinearRegression: средняя RMSE **571.98**;
* RandomForestRegressor: средняя RMSE **369.46**.

---

### Общий вывод по Шагу 10

RandomForest значительно лучше линейной регрессии и не даёт отрицательных предсказаний. Дальше пробуем отдельные модели для каждого таргета.

## Шаг 11. Отдельные RandomForest-модели для каждого таргета

Теперь пробуем другой вариант: обучить отдельную модель для каждого таргета.



### Шаг 11.1. Модель для IC50

In [ ]:
rf_ic50 = RandomForestRegressor(
    n_estimators=300,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_ic50.fit(X_train_filled, y_train["IC50, mM"])

pred_ic50 = rf_ic50.predict(X_valid_filled)

score_ic50_sep = rmse(y_valid["IC50, mM"], pred_ic50)

print("RMSE для IC50:")
print(f"{score_ic50_sep:.5f}")

#### Вывод

RMSE для `IC50`: **388.876**

### Шаг 11.2. Модель для CC50

In [ ]:
rf_cc50 = RandomForestRegressor(
    n_estimators=300,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_cc50.fit(X_train_filled, y_train["CC50, mM"])

pred_cc50 = rf_cc50.predict(X_valid_filled)

score_cc50_sep = rmse(y_valid["CC50, mM"], pred_cc50)

print("RMSE для CC50:")
print(f"{score_cc50_sep:.5f}")

#### Вывод

RMSE для `CC50`: **429.289**

### Шаг 11.3. Модель для SI

`SI` предсказывается отдельной моделью.

In [ ]:
rf_si = RandomForestRegressor(
    n_estimators=300,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_si.fit(X_train_filled, y_train["SI"])

pred_si = rf_si.predict(X_valid_filled)

score_si_sep = rmse(y_valid["SI"], pred_si)

print("RMSE для SI:")
print(f"{score_si_sep:.5f}")

#### Вывод

RMSE для `SI`: **296.269**

### Шаг 11.4. Сравниваем все варианты

In [ ]:
rf_sep_predictions = np.column_stack([
    pred_ic50,
    pred_cc50,
    pred_si
])

rf_sep_mean_score = np.mean([
    score_ic50_sep,
    score_cc50_sep,
    score_si_sep
])

print("Отдельные RandomForest-модели validation RMSE:")
print(f"IC50, mM: {score_ic50_sep:.5f}")
print(f"CC50, mM: {score_cc50_sep:.5f}")
print(f"SI: {score_si_sep:.5f}")
print(f"Средняя RMSE: {rf_sep_mean_score:.5f}")

#### Вывод

Отдельные модели: средняя RMSE **371.48** - немного хуже общей (369.46).

Но это ожидаемо: здесь мы ещё не применяли логарифмирование и очистку признаков.

### Шаг 11.5. Проверяем отрицательные предсказания

In [ ]:
rf_sep_result = y_valid.copy()

rf_sep_result["pred_IC50"] = rf_sep_predictions[:, 0]
rf_sep_result["pred_CC50"] = rf_sep_predictions[:, 1]
rf_sep_result["pred_SI"] = rf_sep_predictions[:, 2]

rf_sep_pred_cols = ["pred_IC50", "pred_CC50", "pred_SI"]

print("Минимальные предсказанные значения:")
print(rf_sep_result[rf_sep_pred_cols].min())

print("\nКоличество отрицательных предсказаний:")
print((rf_sep_result[rf_sep_pred_cols] < 0).sum())

#### Вывод

Отрицательных предсказаний нет ни у одного таргета.

### Шаг 11.6. Итоговая таблица сравнения

In [ ]:
models_result = pd.DataFrame({
    "model": [
        "LinearRegression",
        "RandomForestRegressor",
        "RandomForestRegressor_separate"
    ],
    "IC50_RMSE": [
        score_ic50_base,
        score_ic50,
        score_ic50_sep
    ],
    "CC50_RMSE": [
        score_cc50_base,
        score_cc50,
        score_cc50_sep
    ],
    "SI_RMSE": [
        score_si_base,
        score_si,
        score_si_sep
    ],
    "mean_RMSE": [
        mean_rmse,
        rf_mean_score,
        rf_sep_mean_score
    ]
})

models_result = models_result.sort_values("mean_RMSE")
models_result

#### Вывод

Общая модель `RandomForestRegressor` пока лучше отдельных: 369.46 против 371.48.

---

### Общий вывод по Шагу 11

Мы проверили два варианта RandomForest:
* общая модель: средняя RMSE **369.46**;
* отдельные модели: средняя RMSE **371.48**.

Отдельные модели в базовом варианте немного хуже. Но подход важен - именно он позволяет применять разные преобразования для разных таргетов (например, логарифмирование для SI). Продолжаем улучшение.

## Шаг 12. Небольшой подбор параметров RandomForestRegressor

Проверяем несколько вариантов параметров:
* `n_estimators` - количество деревьев;
* `max_features` - доля признаков для каждого дерева;
* `min_samples_leaf` - минимальное число объектов в листе;
* `max_depth` - максимальная глубина дерева.

### Шаг 12.1. Задаём варианты параметров

In [ ]:
rf_variants = [
    ("rf_300", 300, 1/3, 5, None),
    ("rf_500", 500, 1/3, 5, None),
    ("rf_700", 700, 1/3, 5, None),
    ("rf_leaf_3", 500, 1/3, 3, None),
    ("rf_depth_20", 500, 1/3, 5, 20),
    ("rf_features_05", 500, 0.5, 5, None),
]

print("Задано вариантов:", len(rf_variants))

#### Вывод

Задано несколько вариантов с разным числом деревьев и другими параметрами.

### Шаг 12.2. Обучаем и сравниваем варианты

In [ ]:
rf_tuning_results = []

for name, trees, features, leaf, depth in rf_variants:
    model = RandomForestRegressor(
        n_estimators=trees,
        max_features=features,
        min_samples_leaf=leaf,
        max_depth=depth,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_filled, y_train)
    predictions = model.predict(X_valid_filled)

    ic50 = rmse(y_valid["IC50, mM"], predictions[:, 0])
    cc50 = rmse(y_valid["CC50, mM"], predictions[:, 1])
    si = rmse(y_valid["SI"], predictions[:, 2])
    mean_score = np.mean([ic50, cc50, si])

    rf_tuning_results.append([name, trees, features, leaf, depth, ic50, cc50, si, mean_score])

rf_tuning_results = pd.DataFrame(
    rf_tuning_results,
    columns=[
        "model",
        "n_estimators",
        "max_features",
        "min_samples_leaf",
        "max_depth",
        "IC50_RMSE",
        "CC50_RMSE",
        "SI_RMSE",
        "mean_RMSE"
    ]
)

rf_tuning_results = rf_tuning_results.sort_values("mean_RMSE")
rf_tuning_results

#### Вывод

Лучший вариант - `rf_700` со средней RMSE **365.80**.

### Шаг 12.3. Сохраняем лучший вариант

In [ ]:
best_rf_params = rf_tuning_results.iloc[0]

best_rf_params

#### Вывод

Лучший вариант: `n_estimators=700`, `max_features=1/3`, `min_samples_leaf=5`, `max_depth=None`.

### Шаг 12.4. Финальная проверка лучшего варианта

In [ ]:
best_rf_model = RandomForestRegressor(
    n_estimators=700,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

best_rf_model.fit(X_train_filled, y_train)

best_rf_predictions = best_rf_model.predict(X_valid_filled)

best_rf_ic50 = rmse(y_valid["IC50, mM"], best_rf_predictions[:, 0])
best_rf_cc50 = rmse(y_valid["CC50, mM"], best_rf_predictions[:, 1])
best_rf_si = rmse(y_valid["SI"], best_rf_predictions[:, 2])

best_rf_mean = np.mean([best_rf_ic50, best_rf_cc50, best_rf_si])

print("Лучший RandomForestRegressor validation RMSE:")
print(f"IC50, mM: {best_rf_ic50:.5f}")
print(f"CC50, mM: {best_rf_cc50:.5f}")
print(f"SI: {best_rf_si:.5f}")
print(f"Средняя RMSE: {best_rf_mean:.5f}")

#### Вывод

Модель `rf_700` проверена на validation:
* `IC50, mM`: 384.08, `CC50, mM`: 430.75, `SI`: 282.56;
* **средняя RMSE: 365.80**.

### Шаг 12.5. Проверяем отрицательные предсказания

In [ ]:
best_rf_result = y_valid.copy()

best_rf_result["pred_IC50"] = best_rf_predictions[:, 0]
best_rf_result["pred_CC50"] = best_rf_predictions[:, 1]
best_rf_result["pred_SI"] = best_rf_predictions[:, 2]

pred_cols = ["pred_IC50", "pred_CC50", "pred_SI"]

print("Минимальные предсказанные значения:")
print(best_rf_result[pred_cols].min())

print("\nКоличество отрицательных предсказаний:")
print((best_rf_result[pred_cols] < 0).sum())

#### Вывод

Отрицательных предсказаний нет.

### Шаг 12.6. Обновляем таблицу сравнения

In [ ]:
models_result = pd.DataFrame({
    "model": [
        "LinearRegression",
        "RandomForestRegressor",
        "RandomForestRegressor_separate",
        "RandomForestRegressor_tuned"
    ],
    "IC50_RMSE": [
        score_ic50_base,
        score_ic50,
        score_ic50_sep,
        best_rf_ic50
    ],
    "CC50_RMSE": [
        score_cc50_base,
        score_cc50,
        score_cc50_sep,
        best_rf_cc50
    ],
    "SI_RMSE": [
        score_si_base,
        score_si,
        score_si_sep,
        best_rf_si
    ],
    "mean_RMSE": [
        mean_rmse,
        rf_mean_score,
        rf_sep_mean_score,
        best_rf_mean
    ]
})

models_result = models_result.sort_values("mean_RMSE")
models_result

#### Вывод

Лучший результат - `RandomForestRegressor_tuned`: средняя RMSE **365.80**.

---

### Общий вывод по Шагу 12

Небольшой подбор параметров помог:
* обычный RandomForest: средняя RMSE **369.46**;
* `rf_700` (tuned): средняя RMSE **365.80**.

Дальше используем эти параметры как основу.

## Шаг 13. Создаём submission с RandomForestRegressor_tuned

Обучаем лучшую модель на всех train-данных и создаём submission.

`SI` предсказывается моделью - не формулой.

### Шаг 13.1. Обучаем модель на всём train

In [ ]:
from sklearn.impute import SimpleImputer

final_imputer = SimpleImputer(strategy="median")

X_filled = final_imputer.fit_transform(X)
X_test_filled = final_imputer.transform(X_test)

final_model = RandomForestRegressor(
    n_estimators=700,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_model.fit(X_filled, y)

test_predictions = final_model.predict(X_test_filled)

print("Размер предсказаний:", test_predictions.shape)

#### Вывод

Модель обучена на 751 объекте. Предсказания для test: `(250, 3)`.

### Шаг 13.2. Проверяем предсказания для test

In [ ]:
test_predictions_df = pd.DataFrame(
    test_predictions,
    columns=submission_target_cols
)

display(test_predictions_df.head())

print("Размер test_predictions_df:", test_predictions_df.shape)

print("\nМинимальные предсказанные значения:")
print(test_predictions_df.min())

print("\nКоличество отрицательных предсказаний:")
print((test_predictions_df < 0).sum())

print("\nКоличество пропусков:")
print(test_predictions_df.isna().sum())

#### Вывод

* Размер: `(250, 3)`;
* Отрицательных значений нет;
* Пропусков нет.

Предсказания выглядят корректно.

### Шаг 13.3. Создаём submission-файл

In [ ]:
submission = sample_submission.copy()

submission["IC50"] = test_predictions[:, 0]
submission["CC50"] = test_predictions[:, 1]
submission["SI"] = test_predictions[:, 2]

display(submission.head())

print("Размер submission:", submission.shape)
print("Колонки submission:", submission.columns.tolist())

assert submission.shape == sample_submission.shape
assert submission.columns.tolist() == sample_submission.columns.tolist()

print("Формат submission совпадает с sample_submission")

#### Вывод

Файл создан. Размер: `(250, 4)`, формат совпадает с `sample_submission`.

### Шаг 13.4. Сохраняем submission

In [ ]:
from pathlib import Path

submissions_dir = Path("submissions")
submissions_dir.mkdir(exist_ok=True)

output_path = submissions_dir / "submission_random_forest_tuned.csv"

submission.to_csv(output_path, index=False)

print("Файл сохранён:")
print(output_path)

### Общий вывод по Шагу 13

Мы создали submission с `RandomForestRegressor_tuned`.

Результат на validation:
* `IC50`: 384.084, `CC50`: 430.750, `SI`: 282.561;
* **средняя RMSE: 365.798**.

Это лучше baseline (571.977) и обычного RandomForest (369.463).
Берём эти параметры как основу для следующих шагов.

## Шаг 14. Удаление сильно коррелирующих признаков

В данных 210 признаков, часть из них может повторять друг друга - высокая корреляция между признаками означает, что они несут одинаковую информацию.

Удаляем признаки с корреляцией > 0.95.

**Важно:** список признаков для удаления определяем только по `X_train`. К validation и test применяем тот же список.

### Шаг 14.1. Находим сильно коррелирующие признаки

In [ ]:
corr_matrix = X_train_filled.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

cols_to_drop = [
    col for col in upper_triangle.columns
    if any(upper_triangle[col] > 0.95)
]

print("Количество признаков для удаления:", len(cols_to_drop))

#### Вывод

Найдено 33 сильно коррелирующих признака.

### Шаг 14.2. Удаляем признаки

In [ ]:
X_train_corr = X_train_filled.drop(columns=cols_to_drop)
X_valid_corr = X_valid_filled.drop(columns=cols_to_drop)

print("Было признаков:", X_train_filled.shape[1])
print("Стало признаков:", X_train_corr.shape[1])

#### Вывод

После удаления осталось 177 признаков (было 210).

### Шаг 14.3. Проверяем модель на очищенных признаках

In [ ]:
rf_corr_model = RandomForestRegressor(
    n_estimators=700,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_corr_model.fit(X_train_corr, y_train)

rf_corr_predictions = rf_corr_model.predict(X_valid_corr)

rf_corr_ic50 = rmse(y_valid["IC50, mM"], rf_corr_predictions[:, 0])
rf_corr_cc50 = rmse(y_valid["CC50, mM"], rf_corr_predictions[:, 1])
rf_corr_si = rmse(y_valid["SI"], rf_corr_predictions[:, 2])

rf_corr_mean = np.mean([rf_corr_ic50, rf_corr_cc50, rf_corr_si])

print("RandomForest после удаления коррелирующих признаков:")
print(f"IC50, mM: {rf_corr_ic50:.5f}")
print(f"CC50, mM: {rf_corr_cc50:.5f}")
print(f"SI: {rf_corr_si:.5f}")
print(f"Средняя RMSE: {rf_corr_mean:.5f}")

#### Вывод

После удаления коррелирующих признаков:
* `IC50, mM`: 383.39, `CC50, mM`: 424.44, `SI`: 282.11;
* **средняя RMSE: 363.32** — улучшилась с 365.80.

---

### Общий вывод по Шагу 14

Удаление 33 коррелирующих признаков немного улучшило модель: RMSE 365.80 => 363.32.

## Шаг 15. Удаление почти константных признаков

Некоторые признаки почти не меняются - если одно значение встречается в > 99% строк, такой признак мало полезен для модели.

Удаляем такие признаки.

### Шаг 15.1. Находим почти константные признаки

In [ ]:
constant_cols = []

for col in X_train_corr.columns:
    value_counts = X_train_corr[col].value_counts(normalize=True)

    if value_counts.iloc[0] > 0.99:
        constant_cols.append(col)

print("Найдено почти константных признаков:", len(constant_cols))

#### Вывод

Найдено 34 почти константных признака.

### Шаг 15.2. Удаляем признаки

In [ ]:
X_train_clean = X_train_corr.drop(columns=constant_cols)
X_valid_clean = X_valid_corr.drop(columns=constant_cols)

print("Признаков после удаления коррелирующих:", X_train_corr.shape[1])
print("Признаков после удаления почти константных:", X_train_clean.shape[1])

#### Вывод

После удаления осталось 143 признака (было 177).

### Шаг 15.3. Проверяем модель

In [ ]:
rf_clean_model = RandomForestRegressor(
    n_estimators=700,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_clean_model.fit(X_train_clean, y_train)

rf_clean_predictions = rf_clean_model.predict(X_valid_clean)

rf_clean_ic50 = rmse(y_valid["IC50, mM"], rf_clean_predictions[:, 0])
rf_clean_cc50 = rmse(y_valid["CC50, mM"], rf_clean_predictions[:, 1])
rf_clean_si = rmse(y_valid["SI"], rf_clean_predictions[:, 2])

rf_clean_mean = np.mean([rf_clean_ic50, rf_clean_cc50, rf_clean_si])

print("RandomForest после полной очистки признаков:")
print(f"IC50, mM: {rf_clean_ic50:.5f}")
print(f"CC50, mM: {rf_clean_cc50:.5f}")
print(f"SI: {rf_clean_si:.5f}")
print(f"Средняя RMSE: {rf_clean_mean:.5f}")

#### Вывод

После удаления почти константных признаков:
* `IC50, mM`: 383.91, `CC50, mM`: 424.23, `SI`: 279.71;
* **средняя RMSE: 362.62** — ещё немного улучшилась.

---

### Общий вывод по Шагу 15

После двух шагов очистки признаков:
* удалено 33 коррелирующих + 34 почти константных = **67 признаков**;
* осталось 143 признака.

## Шаг 16. Отбор признаков по важности

У `RandomForestRegressor` есть встроенная оценка важности признаков - `feature_importances_`.

Проверяем, поможет ли удаление признаков с маленькой важностью.

### Шаг 16.1. Получаем важности признаков

In [ ]:
feature_importance = pd.Series(
    rf_clean_model.feature_importances_,
    index=X_train_clean.columns
).sort_values(ascending=False)

display(feature_importance.head(10))

#### Вывод

Самые важные признаки: `VSA_EState6`, `BalabanJ`, `Kappa3`, `Ipc`, `MolWt`.

### Шаг 16.2. Оставляем признаки с важностью > 0.002

In [ ]:
important_cols = feature_importance[feature_importance > 0.002].index.tolist()

X_train_imp = X_train_clean[important_cols]
X_valid_imp = X_valid_clean[important_cols]

print("Было признаков:", X_train_clean.shape[1])
print("Осталось признаков:", len(important_cols))

#### Вывод

После отбора осталось 76 признаков (было 143).

### Шаг 16.3. Проверяем модель на отобранных признаках

In [ ]:
rf_imp_model = RandomForestRegressor(
    n_estimators=700,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_imp_model.fit(X_train_imp, y_train)

rf_imp_predictions = rf_imp_model.predict(X_valid_imp)

rf_imp_ic50 = rmse(y_valid["IC50, mM"], rf_imp_predictions[:, 0])
rf_imp_cc50 = rmse(y_valid["CC50, mM"], rf_imp_predictions[:, 1])
rf_imp_si = rmse(y_valid["SI"], rf_imp_predictions[:, 2])

rf_imp_mean = np.mean([rf_imp_ic50, rf_imp_cc50, rf_imp_si])

print("RandomForest после отбора по важности:")
print(f"IC50, mM: {rf_imp_ic50:.5f}")
print(f"CC50, mM: {rf_imp_cc50:.5f}")
print(f"SI: {rf_imp_si:.5f}")
print(f"Средняя RMSE: {rf_imp_mean:.5f}")

#### Вывод

После отбора по важности:
* **средняя RMSE: 362.97** - чуть хуже, чем с 143 признаками (362.62).

---

### Общий вывод по Шагу 16

Отбор по `feature_importances` не улучшил результат:
* до отбора: RMSE **362.62**;
* после отбора: RMSE **362.97**.

Оставляем 143 признака и продолжаем с ними.

## Шаг 17. Масштабирование признаков с помощью StandardScaler

Масштабирование признаков через `StandardScaler` приведёт все признаки к одному масштабу.

**Важно:** `StandardScaler` обучаем только на `X_train_clean`. К validation применяем только `.transform()` - без повторного `.fit()`.

### Шаг 17.1. Применяем StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_clean)
X_valid_scaled = scaler.transform(X_valid_clean)

print("Размер X_train_scaled:", X_train_scaled.shape)
print("Размер X_valid_scaled:", X_valid_scaled.shape)

#### Вывод

Признаки масштабированы. Размеры не изменились: 600×143 и 151×143.

### Шаг 17.2. Проверяем RandomForest на масштабированных признаках

In [ ]:
rf_scaled_model = RandomForestRegressor(
    n_estimators=700,
    max_features=1/3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_scaled_model.fit(X_train_scaled, y_train)

rf_scaled_predictions = rf_scaled_model.predict(X_valid_scaled)

rf_scaled_ic50 = rmse(y_valid["IC50, mM"], rf_scaled_predictions[:, 0])
rf_scaled_cc50 = rmse(y_valid["CC50, mM"], rf_scaled_predictions[:, 1])
rf_scaled_si = rmse(y_valid["SI"], rf_scaled_predictions[:, 2])

rf_scaled_mean = np.mean([rf_scaled_ic50, rf_scaled_cc50, rf_scaled_si])

print("RandomForest после масштабирования:")
print(f"IC50, mM: {rf_scaled_ic50:.5f}")
print(f"CC50, mM: {rf_scaled_cc50:.5f}")
print(f"SI: {rf_scaled_si:.5f}")
print(f"Средняя RMSE: {rf_scaled_mean:.5f}")

#### Вывод

После масштабирования:
* `IC50, mM`: 382.63, `CC50, mM`: 422.41, `SI`: 280.69;
* **средняя RMSE: 361.91** - лучший результат на данный момент.

---

### Общий вывод по Шагу 17

Масштабирование немного улучшило результат: 362.62 => 361.91.

Итоговый набор признаков для финального обучения: **143 признака + StandardScaler**.